# <center>Modelling</center>

In [71]:
# importing the libraries
import pandas as pd
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, RMSprop
import keras_tuner as kt
import numpy as np
from sklearn.utils import class_weight
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [72]:
# Importing the csv file
df = pd.read_csv("data/df.csv")

In [73]:
# Checking the shape
df.shape

(10000, 15)

In [74]:
# Checking the first five rows of the dataset
df.head()

,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,balance_per_product,engagement_score,age_group,zero_balance
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0.00,1,"(30, 45]",1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,41903.93,1,"(30, 45]",0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,39915.20,0,"(30, 45]",0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.00,0,"(30, 45]",1
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,62755.41,1,"(30, 45]",0


In [75]:
df['age_group'].value_counts()

age_group
(30, 45]     5921
(18, 30]     1968
(45, 60]     1647
(60, 100]     464
Name: count, dtype: int64

In [76]:
# train test split
X = df.drop("exited", axis=1)
y = df["exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 14), (2000, 14), (8000,), (2000,))

In [77]:
# Preprocessing the data
num_cols = [col for col in df.select_dtypes(include='number').columns if df[col].nunique() > 2]

binary_cols = [col for col in df.select_dtypes(include='number').columns if df[col].nunique() == 2]

if "exited" in binary_cols:
    binary_cols.remove("exited")
cat_cols = [col for col in df.select_dtypes(include='object').columns]


preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols),
        ('bin', 'passthrough', binary_cols)
    ]
)

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [83]:
# Storing the results in a list

results = []

# Creating a function to add results to the list
def evaluate_model(name, technique, model, X_test, y_test, threshold=0.5):
    
    y_prob = model.predict(X_test).ravel()
    y_pred = (y_prob > threshold).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "model": name,
        "technique": technique,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    })
    
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("AUC-ROC:", auc)

## <center>Baseline Model</center>

In [49]:
# Baseline Model
model_baseline = Sequential()
model_baseline.add(Input(shape=(X_train.shape[1],)))
model_baseline.add(Dense(32, activation='relu'))
model_baseline.add(Dense(1, activation='sigmoid'))

# model summary
model_baseline.summary()

# Compiling the model
model_baseline.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# fitting the model
history = model_baseline.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[early_stop])

# Model Evaluation
evaluate_model("model", 'Baseline', model_baseline , X_test, y_test, threshold=0.5)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 32)             │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7788 - loss: 0.4966 - val_accuracy: 0.8169 - val_loss: 0.4347
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8178 - loss: 0.4189 - val_accuracy: 0.8431 - val_loss: 0.4059
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8297 - loss: 0.3969 - val_accuracy: 0.8500 - val_loss: 0.3837
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8388 - loss: 0.3807 - val_accuracy: 0.8581 - val_loss: 0.3671
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8417 - loss: 0.3684 - val_accuracy: 0.8650 - val_loss: 0.3540
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8495 - loss: 0.3591 - val_accuracy: 0.8687 - val_loss: 0.3457
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8527 - loss: 0.3531 - val_accuracy: 0.8656 - val_loss: 0.3390
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8533 - loss: 0.3490 - val_accu

### Insights
- The model achieves 87% accuracy, indicating strong overall performance across the dataset.
- AUC-ROC = 0.86, showing the model has a strong ability to distinguish between churn and non-churn customers.
- Recall (churn) = 49%, meaning the model is missing more than half of actual churn customers (213 missed cases).
- False negatives = 206, which is a key weakness since many churn customers are not being identified.
- Precision (churn) = 77%, indicating that when the model predicts churn, it is usually correct (low false positives).
- False positives = 59, which is relatively low, making the model conservative in predicting churn.
- The model is biased toward predicting non-churn, with very high performance for class 0 (96% recall).
- There is a clear imbalance in performance, where the model favors precision over recall for the churn class.

In [50]:
# Threshold Tuning
evaluate_model("model", 'Threshold', model_baseline , X_test, y_test, threshold=0.3)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[[1410  183]
 [ 134  273]]
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      1593
           1       0.60      0.67      0.63       407

    accuracy                           0.84      2000
   macro avg       0.76      0.78      0.77      2000
weighted avg       0.85      0.84      0.84      2000

AUC-ROC: 0.8638607791150165


### Insights:
- Model accuracy is 85%, slightly lower than baseline, but this is expected after adjusting the decision boundary.
- Churn recall improved to 0.67, meaning the model is now catching more churn customers compared to baseline (better business impact).
- Churn precision dropped to 0.62, indicating more false alarms (non-churn customers being predicted as churn).
- False negatives reduced (136), which is a positive improvement because fewer churn customers are being missed.
- Class 0 performance slightly reduced (recall 0.90), meaning some non-churn customers are now being misclassified.
- AUC remains strong (0.865), showing that the model’s ability to rank churn vs non-churn is unchanged—only the decision threshold changed behavior.
- The model is now more sensitive toward churn detection, shifting trade-off from precision to recall.

In [52]:
X_train.shape[1]

18

In [84]:
# Class Weighting

# Calculating class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))

# Building the model with class weights
model_weighted = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Compiling the model
model_weighted.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# fitting the model with class weights
history_weighted = model_weighted.fit(X_train, y_train, epochs=100, validation_split=0.2, class_weight=class_weights_dict, callbacks=[early_stop])

# Model Evaluation
evaluate_model("model_weighted", 'class_weight', model_weighted , X_test, y_test, threshold=0.5)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6052 - loss: 0.6136 - val_accuracy: 0.7387 - val_loss: 0.5665
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7456 - loss: 0.5397 - val_accuracy: 0.7487 - val_loss: 0.5380
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7516 - loss: 0.5166 - val_accuracy: 0.7850 - val_loss: 0.4804
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7658 - loss: 0.5015 - val_accuracy: 0.7631 - val_loss: 0.4975
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7755 - loss: 0.4895 - val_accuracy: 0.7844 - val_loss: 0.4664
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7800 - loss: 0.4809 - val_accuracy: 0.7831 - val_loss: 0.4630
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7789 - loss: 0.4740 - val_accuracy: 0.8012 - val_loss: 0.4412
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7819 - loss: 0.4698 - val_accu

### Insights:
- Model achieves 81% accuracy, showing a slight drop compared to baseline, but this is expected after handling class imbalance.
- Churn recall improved to 0.75, meaning the model is now correctly identifying more churn customers compared to baseline and threshold-tuned models.
- False negatives reduced to 103, which is a strong improvement and very important for churn prediction business use case.
- Churn precision is 0.52, meaning the model is also producing more false alarms (non-churn predicted as churn).
- False positives = 287, indicating the model is more aggressive in predicting churn.
- Class 0 performance dropped slightly (recall 0.82), meaning some non-churn customers are being misclassified.
- AUC-ROC = 0.86, which is strong and shows the model still has good ability to separate classes despite imbalance handling.

In [ ]:
# SMOTE Oversampling

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

model_sm = Sequential([
    Dense(32, activation='relu', input_dim=X_train.shape[1]),
    Dense(1, activation='sigmoid')
])

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)


model_sm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model_sm.fit(
    X_train_sm, y_train_sm,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

evaluate_model("SMOTE_ANN", 'SMOTE', model_sm , X_test, y_test, threshold=0.5)

Epoch 1/100


d:\DL\bank_customer_churn_prediction\myenv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7158 - loss: 0.5519 - val_accuracy: 0.5914 - val_loss: 0.6957
Epoch 2/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7816 - loss: 0.4719 - val_accuracy: 0.6480 - val_loss: 0.6123
Epoch 3/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7933 - loss: 0.4462 - val_accuracy: 0.6735 - val_loss: 0.5746
Epoch 4/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8002 - loss: 0.4316 - val_accuracy: 0.6915 - val_loss: 0.5546
Epoch 5/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8057 - loss: 0.4238 - val_accuracy: 0.6864 - val_loss: 0.5597
Epoch 6/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8064 - loss: 0.4186 - val_accuracy: 0.6601 - val_loss: 0.6138
Epoch 7/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8057 - loss: 0.4157 - val_accuracy: 0.6660 - val_loss: 0.5863
Epoch 8/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8079 - loss: 0.4136 - val_accuracy: 0.7135

### Insights:
- Model achieves 82% accuracy, which is slightly lower than baseline, but expected due to synthetic balancing of classes.
- Churn recall improved to 0.68, meaning the model is now correctly identifying more churn customers compared to baseline, but slightly lower than class-weight model.
- Churn precision dropped to 0.55, indicating more false positives (non-churn customers incorrectly predicted as churn).
- False negatives (132) are reduced compared to baseline, showing better churn capture, but not as strong as class-weighting.
- Class 0 performance decreased slightly (recall 0.86), meaning some non-churn customers are being misclassified.
- AUC-ROC = 0.858, which is strong and very close to baseline and class-weight model, showing overall ranking ability is stable.
- Training curve shows stable learning but slight validation fluctuations, indicating mild overfitting tendency after oversampling.

In [ ]:
results

[{'model': 'model',
  'technique': 'Baseline',
  'accuracy': 0.8695,
  'precision': 0.7943548387096774,
  'recall': 0.48402948402948404,
  'f1': 0.601526717557252,
  'auc': 0.8619636585738281},
 {'model': 'model',
  'technique': 'Threshold',
  'accuracy': 0.8415,
  'precision': 0.5991189427312775,
  'recall': 0.6683046683046683,
  'f1': 0.6318234610917538,
  'auc': 0.8619636585738281},
 {'model': 'model_weighted',
  'technique': 'class_weight',
  'accuracy': 0.814,
  'precision': 0.5313059033989267,
  'recall': 0.7297297297297297,
  'f1': 0.6149068322981367,
  'auc': 0.8617045396706414},
 {'model': 'SMOTE_ANN',
  'technique': 'SMOTE',
  'accuracy': 0.8225,
  'precision': 0.5522088353413654,
  'recall': 0.6756756756756757,
  'f1': 0.6077348066298343,
  'auc': 0.858342163426909}]

In [ ]:
results_df = pd.DataFrame(results)
results_df

,model,technique,accuracy,precision,recall,f1,auc
0,model,Baseline,0.8695,0.794355,0.484029,0.601527,0.861964
1,model,Threshold,0.8415,0.599119,0.668305,0.631823,0.861964
2,model_weighted,class_weight,0.8140,0.531306,0.729730,0.614907,0.861705
3,SMOTE_ANN,SMOTE,0.8225,0.552209,0.675676,0.607735,0.858342


### Insights:
- Baseline model has highest accuracy (0.87), but it is misleading because it has poor churn recall (0.48), meaning it misses many churn customers.
- Threshold tuning improves churn detection (recall 0.67) while maintaining strong AUC, making it more balanced for business use compared to baseline.
- Class weighting gives the highest churn recall (0.73), meaning it captures the most churn customers, but at the cost of lower precision (more false positives).
- SMOTE model performs similarly to class weights in recall (0.67) but has slightly lower precision and slightly lower AUC, indicating it is less stable for this dataset.
- AUC is almost constant (~0.86) across all models, meaning all models have similar ranking ability, and differences come mainly from threshold/imbalance handling.

In [ ]:
# Keras Tuner Hyperparameter Tuning
def build_model(hp):

    model = keras.Sequential()

    # First layer
    model.add(Dense(
        units=hp.Choice('units_1', [16, 32, 64, 128, 256]),
        activation=hp.Choice('activation_1', ['relu', 'tanh']),
        input_shape=(X_train.shape[1],)
    ))

    # dropout
    model.add(Dropout(rate=hp.Choice('dropout_rate', [0.2, 0.3, 0.4, 0.5])))

    # second layer
    if hp.Boolean('second_layer'):
        model.add(Dense(
            units=hp.Choice('units_2', [16, 32, 64, 128]),
            activation=hp.Choice('activation_1', ['relu', 'tanh'])
        ))

    # dropout
    if hp.Boolean('dropout'):
        model.add(Dropout(rate=hp.Choice('dropout_rate', [0.2, 0.3, 0.4, 0.5])))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Learning rate tuning
    lr = hp.Choice('lr', [1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuning_dir',
    project_name='churn_ann'
)

d:\DL\bank_customer_churn_prediction\myenv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
tuner.search(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Trial 10 Complete [00h 00m 08s]
val_accuracy: 0.8743749856948853

Best val_accuracy So Far: 0.878125011920929
Total elapsed time: 00h 02m 17s


In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

d:\DL\bank_customer_churn_prediction\myenv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\DL\bank_customer_churn_prediction\myenv\lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
evaluate_model("Tuned_model", 'Tuner', best_model , X_test, y_test, threshold=0.5)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
[[1550   43]
 [ 234  173]]
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.80      0.43      0.56       407

    accuracy                           0.86      2000
   macro avg       0.83      0.70      0.74      2000
weighted avg       0.86      0.86      0.84      2000

AUC-ROC: 0.8597426394036564


### Insights:
- Model achieves 86% accuracy with AUC ~0.86, showing that tuning improved stability of the model but not the underlying class separation ability significantly.
- Non-churn class performance is very strong (recall 0.97), meaning the model is highly reliable in identifying customers who will stay.
- Churn class recall dropped to 0.43, which is a significant weakness and means the model is missing more than half of actual churn customers.
- Churn precision is high (0.80), meaning when the model predicts churn, it is usually correct, but it is very conservative in making churn predictions.
- There is a clear bias toward predicting class 0 (non-churn), leading to high false negatives (234 churn customers missed).
- Compared to earlier models, tuning improved structure efficiency slightly, but did not solve imbalance-driven learning issues.

In [ ]:
# Class Weighting on tuned model

# Calculating class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))


# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Compiling the model
best_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# fitting the model with class weights
history_weighted_best_model = best_model.fit(X_train, y_train, epochs=100, validation_split=0.2, class_weight=class_weights_dict, callbacks=[early_stop])

# Model Evaluation
evaluate_model("best_model_weighted", 'class_weight', best_model , X_test, y_test, threshold=0.5)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7952 - loss: 0.4720 - val_accuracy: 0.7756 - val_loss: 0.4484
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7944 - loss: 0.4605 - val_accuracy: 0.7856 - val_loss: 0.4308
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7939 - loss: 0.4533 - val_accuracy: 0.7975 - val_loss: 0.4168
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7945 - loss: 0.4536 - val_accuracy: 0.7981 - val_loss: 0.4252
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7950 - loss: 0.4547 - val_accuracy: 0.7962 - val_loss: 0.4195
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7945 - loss: 0.4468 - val_accuracy: 0.7794 - val_loss: 0.4458
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7975 - loss: 0.4428 - val_accuracy: 0.8019 - val_loss: 0.4176
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7925 - loss: 0.4486 - val_accu

### Insights:
- Model accuracy is 80%, which is lower than baseline/tuned-only models, but this drop is expected because the model is now penalizing minority-class errors more heavily.
- Churn recall improved significantly to 0.72, meaning the model is now correctly identifying most churn customers, which is the primary objective in this problem.
- False negatives reduced to 114, which is a strong improvement and indicates better churn capture compared to previous models.
- Churn precision dropped to 0.51, meaning the model is generating more false positives (predicting churn for non-churn customers incorrectly).
- Class 0 performance reduced (recall 0.82), showing the model is now less conservative toward majority class predictions.
- AUC remains strong (~0.86), meaning the model still has good separation ability between churn and non-churn despite the class imbalance handling.

In [ ]:
results_df = pd.DataFrame(results)
results_df

,model,technique,accuracy,precision,recall,f1,auc
0,model,Baseline,0.8695,0.794355,0.484029,0.601527,0.861964
1,model,Threshold,0.8415,0.599119,0.668305,0.631823,0.861964
2,model_weighted,class_weight,0.8140,0.531306,0.729730,0.614907,0.861705
3,SMOTE_ANN,SMOTE,0.8225,0.552209,0.675676,0.607735,0.858342
4,Tuned_model,Tuner,0.8610,0.776824,0.444717,0.565625,0.865142
5,best_model_weighted,class_weight,0.8155,0.534050,0.732187,0.617617,0.866296
6,Tuned_model,Tuner,0.8650,0.793991,0.454545,0.578125,0.851944
7,best_model_weighted,class_weight,0.8530,0.645244,0.616708,0.630653,0.860477
8,Tuned_model,Tuner,0.8615,0.800926,0.425061,0.555377,0.859743
9,best_model_weighted,class_weight,0.8030,0.511344,0.719902,0.597959,0.860284


In [ ]:
# Final Threshold Tuning on best model

best_model=model_weighted

y_prob = best_model.predict(X_test)

thresholds = [0.2, 0.3, 0.35, 0.4, 0.5]

for t in thresholds:
    print("\n==============================")
    print(f"Threshold: {t}")
    
    y_pred = (y_prob > t).astype(int)
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("AUC:", roc_auc_score(y_test, y_prob))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

Threshold: 0.2
Confusion Matrix:
[[808 785]
 [ 33 374]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.51      0.66      1593
           1       0.32      0.92      0.48       407

    accuracy                           0.59      2000
   macro avg       0.64      0.71      0.57      2000
weighted avg       0.83      0.59      0.63      2000

AUC: 0.8617045396706414

Threshold: 0.3
Confusion Matrix:
[[1017  576]
 [  64  343]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.64      0.76      1593
           1       0.37      0.84      0.52       407

    accuracy                           0.68      2000
   macro avg       0.66      0.74      0.64      2000
weighted avg       0.83      0.68      0.71      2000

AUC: 0.8617045396706414

Threshold: 0.35
Confusion Matrix:
[[1116  477]
 [  73  334]]

Classification Report:
      

### Insights:
- Lower threshold (0.2–0.3) heavily increases churn recall (0.84–0.92), but:

Accuracy drops significantly (0.59–0.68)

Too many false positives (business noise increases sharply)
- Mid threshold (0.35–0.4) gives the best balance:

Recall stays strong (0.79–0.82)

Precision improves compared to lower thresholds

Overall F1-score becomes more stable
- Higher threshold (0.5) improves precision and accuracy:

Precision = 0.53 (best among all)

But recall drops to 0.73 → more churn cases missed
- AUC remains constant (~0.862):

Confirms model ranking ability is stable

Only decision boundary (threshold) is changing behavior

# <center>Final Full Pipeline Code</center>

In [85]:
# final full pipeline with best model and best threshold


# Calculating class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))

# Building the model with class weights
final_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Compiling the model
final_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# fitting the model with class weights
history_weighted = final_model.fit(X_train, y_train, epochs=100, validation_split=0.2, class_weight=class_weights_dict, callbacks=[early_stop])

# Model Evaluation
evaluate_model("model_weighted", 'class_weight', final_model , X_test, y_test, threshold=0.5)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6898 - loss: 0.6070 - val_accuracy: 0.7188 - val_loss: 0.5726
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7428 - loss: 0.5468 - val_accuracy: 0.7181 - val_loss: 0.5558
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7455 - loss: 0.5235 - val_accuracy: 0.7425 - val_loss: 0.5240
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7533 - loss: 0.5082 - val_accuracy: 0.7825 - val_loss: 0.4700
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7706 - loss: 0.4959 - val_accuracy: 0.7531 - val_loss: 0.4997
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7692 - loss: 0.4869 - val_accuracy: 0.7812 - val_loss: 0.4704
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7755 - loss: 0.4794 - val_accuracy: 0.7800 - val_loss: 0.4689
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7780 - loss: 0.4739 - val_accu

In [87]:
import joblib

# Save preprocessing
joblib.dump(preprocessor, "preprocessor.pkl")

# Save model
best_model = final_model
best_model.save("churn_model.keras")

final_threshold = 0.35